#  Script 3 — Labeling + Naive Bayes + Evaluasi
## Analisis Sentimen Konten Dakwah Penceramah Kota Bandung
**Author:** Fahmi Faturahman — STMIK Bandung 2026

---
###  Alur Script:
```
CELL 1 → Install library
CELL 2 → Mount Google Drive & Load data bersih
CELL 3 → Labeling Semi-Otomatis (Lexicon-Based)
CELL 4 → Cek & validasi distribusi label
CELL 5 → Preprocessing lanjutan (Tokenizing + Stemming)
CELL 6 → TF-IDF Feature Extraction
CELL 7 → Split data Train / Test
CELL 8 → Training Multinomial Naive Bayes
CELL 9 → Evaluasi: Confusion Matrix + Classification Report
CELL 10 → Visualisasi hasil evaluasi
CELL 11 → Simpan model & hasil
```
### Input & Output:
- **Input:**  `clean_data_komentar.csv` (output dari Script 2)
- **Output:** `model_naive_bayes.pkl` + `hasil_prediksi.csv` + grafik evaluasi

>  **Jalankan Script 1 dan Script 2 terlebih dahulu!**

---
## Cell 1 — Install Library
> Jalankan sekali di setiap sesi Colab baru.

In [13]:
!pip install -q PySastrawi
!pip install -q scikit-learn
!pip install -q pandas numpy matplotlib seaborn
!pip install -q openpyxl
# 🆕 BARU — Untuk download InSet Lexicon dari GitHub
!pip install -q requests

print('✅ Semua library berhasil diinstall!')

✅ Semua library berhasil diinstall!


---
## Cell 2 — Mount Google Drive & Load Data
> Load data bersih hasil dari Script 2.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE_DIR       = '/content/drive/MyDrive/skripsi_dakwah_bandung'
INPUT_CSV      = os.path.join(BASE_DIR, 'clean_data_komentar.csv')
OUTPUT_DIR     = os.path.join(BASE_DIR, 'hasil_model')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🆕 BARU — Path checkpoint stemming (resume otomatis jika Colab disconnect)
PATH_STEMMED   = os.path.join(BASE_DIR, 'clean_data_stemmed.csv')

# 🆕 BARU — Path cache InSet Lexicon (download sekali, disimpan di Drive)
PATH_INSET_POS = os.path.join(BASE_DIR, 'inset_lexicon_positive.tsv')
PATH_INSET_NEG = os.path.join(BASE_DIR, 'inset_lexicon_negative.tsv')

# Load data
df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')

print('='*55)
print(' DATA YANG DIMUAT')
print('='*55)
print(f'Total komentar     : {len(df):,}')
print(f'Kolom tersedia     : {list(df.columns)}')
print()
print('5 baris pertama:')
display(df[['comment_text', 'comment_clean']].head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 DATA YANG DIMUAT
Total komentar     : 18,905
Kolom tersedia     : ['comment_id', 'comment_text', 'author_name', 'like_count', 'published_at', 'video_id', 'channel_name', 'topik_utama', 'panjang_raw', 'jumlah_kata_raw', 'comment_clean', 'comment_no_stopword', 'panjang_clean', 'jumlah_kata_clean', 'tokens']

5 baris pertama:


,comment_text,comment_clean
0,"Andai saja hadir juga : Aa Gym, Buya Yahya, Ha...",andai saja hadir juga aa gym buya yahya habib ...
1,Saya terlahir NU dan tumbuh dg segala aturan d...,saya terlahir nu dan tumbuh dg segala aturan d...
2,YANG NONTON INI SAMBIL NANGISS😭☝🏻☝🏻☝🏻 Allahumm...,yang nonton ini sambil nangiss allahumma sholl...
3,saya bersaksi guru guru kami adalah orang oran...,saya bersaksi guru guru kami adalah orang oran...
4,Teringat caption pada foto para guru ini bbrp ...,teringat caption pada foto para guru ini bbrp ...


---
## Cell 3 —  Labeling Semi-Otomatis (InSet Lexicon)

> ### Cara kerja labeling:
> Script ini menggunakan **InSet Lexicon** (Fachrina et al., 2016) sebagai kamus sentimen utama,
> diperkuat dengan ekstensi kosakata domain dakwah Islam.
>
> **Sumber kamus:**
> - **InSet Lexicon** — ~3.600 kata positif + ~6.600 kata negatif (total ~10.218 kata)
>   Diunduh otomatis dari GitHub: `Adaptive-Informatics/InSet`
>   Di-cache ke Google Drive agar tidak perlu download ulang setiap sesi
> - **Ekstensi Dakwah** — kosakata khas konten ceramah yang tidak ada di InSet
>   (masyaallah, subhanallah, barakallah, sesat, bidah, dll)
>
> **Fitur tambahan:**
> -  Deteksi negasi — `"tidak bagus"` → Negatif (bukan Positif)
> -  Lookup O(1) menggunakan Python `set`
> -  Overlap kata dihapus otomatis (kata ambigu tidak dipakai)
>
> **Referensi:**
> Fachrina, W. O., Widyantoro, D. H., & Suhartono, D. (2016).
> *Indonesian Sentiment Lexicon (InSet).*

In [15]:
import requests, os, re
import pandas as pd

# ============================================================
# KAMUS SENTIMEN — InSet Lexicon (Fachrina et al., 2016)
# 🔧 DIPERBAIKI — Dari kamus manual 127 kata → InSet ~10.218 kata
#
# Referensi:
#   Fachrina, W. O., et al. (2016).
#   "Indonesian Sentiment Lexicon."
#   GitHub: Adaptive-Informatics/InSet
#
# Struktur InSet Lexicon (.tsv):
#   Kolom 1: kata
#   Kolom 2: bobot sentimen (positif: +1 s/d +5, negatif: -5 s/d -1)
# ============================================================

# ── URL GitHub InSet Lexicon (raw file) ──────────────────────
INSET_URL_POS = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"

INSET_URL_NEG = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"


def _download_inset(url: str, path_cache: str, label: str) -> pd.DataFrame:
    """
    Download file InSet Lexicon dari GitHub, simpan ke Drive sebagai cache.
    Jika file cache sudah ada → load langsung tanpa download ulang.

    Returns:
        DataFrame dengan kolom ['kata', 'bobot']
    """
    if os.path.exists(path_cache):
        print(f'  ⚡ {label}: Load dari cache Drive ({path_cache.split("/")[-1]})')
        return pd.read_csv(path_cache, sep='\t', header=None,
                           names=['kata', 'bobot'], encoding='utf-8')

    print(f'  ⬇️  {label}: Downloading dari GitHub...')
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()

        # Simpan ke Drive sebagai cache (tidak perlu download lagi sesi berikutnya)
        with open(path_cache, 'w', encoding='utf-8') as f:
            f.write(resp.text)

        df = pd.read_csv(path_cache, sep='\t', header=None,
                         names=['kata', 'bobot'], encoding='utf-8')
        print(f'     ✅ {len(df):,} kata berhasil dimuat dan disimpan ke Drive')
        return df

    except Exception as e:
        print(f'     ❌ Gagal download {label}: {e}')
        print(f'        Pastikan Colab terhubung internet!')
        return pd.DataFrame(columns=['kata', 'bobot'])


# ── Load InSet Lexicon ────────────────────────────────────────
print('=' * 60)
print('📚 MEMUAT InSet Lexicon (Fachrina et al., 2016)')
print('=' * 60)

df_pos = _download_inset(INSET_URL_POS, PATH_INSET_POS, 'Positive Lexicon')
df_neg = _download_inset(INSET_URL_NEG, PATH_INSET_NEG, 'Negative Lexicon')

# Konversi ke set kata (lowercase) untuk lookup O(1)
INSET_POSITIF = set(df_pos['kata'].str.lower().str.strip().dropna())
INSET_NEGATIF = set(df_neg['kata'].str.lower().str.strip().dropna())

print()
print(f'  Kata Positif (InSet) : {len(INSET_POSITIF):,} kata')
print(f'  Kata Negatif (InSet) : {len(INSET_NEGATIF):,} kata')
print(f'  Total                : {len(INSET_POSITIF) + len(INSET_NEGATIF):,} kata')
print()


# ── Ekstensi Domain Dakwah ────────────────────────────────────
# Kosakata religius khas konten dakwah yang tidak ada di InSet
# (InSet dibuat dari koran/berita, bukan teks dakwah Islam)
DAKWAH_POSITIF = {
    # Ekspresi kagum/syukur Islami — variasi ejaan yang umum di komentar YouTube
    'masyaallah', 'masya allah', 'maa syaa allah', 'maasyaallah',
    'subhanallah', 'subhanalah', 'subhanalloh',
    'alhamdulillah', 'alhamdulilah', 'alhamdullilah',
    'allahuakbar', 'allahu akbar', 'allahu akbar',
    'barakallah', 'barakallahu', 'barakallahu fik', 'barakallahu fiikum',
    'jazakallah', 'jazakallahu', 'jazakallahu khair', 'jazaakallah',
    'aamiin', 'amiin', 'aameen', 'amin',
    # Kosakata positif khas dakwah
    'berkah', 'barokah', 'hidayah', 'taufik', 'rahmat',
    'syukur', 'bersyukur', 'nikmat', 'karunia',
    'ikhlas', 'taqwa', 'bertaqwa', 'amanah',
    'soleh', 'sholeh', 'sholehah', 'solehah',
    'bermanfaat', 'manfaatnya', 'berkesan',
    # Pujian khas komentar dakwah YouTube
    'nyimak', 'hadir', 'menyimak', 'menyimaknya',
    'lanjutkan', 'teruskan', 'istiqomah', 'istiqamah',
    'semangat berdakwah', 'dakwah', 'kajian bagus',
}

DAKWAH_NEGATIF = {
    # Penolakan konten dakwah
    'sesat', 'menyesatkan', 'disesatkan', 'kesesatan',
    'bid ah', 'bidah', "bid'ah", 'bidah',
    'syirik', 'syiriknya', 'musyrik',
    'fitnah', 'memfitnah', 'difitnah',
    'kafir', 'mengkafirkan', 'dikafirkan',
    # Ujaran kebencian di komentar dakwah
    'hujat', 'menghujat', 'penghujat',
    'caci', 'mencaci', 'cacian', 'makian',
    'provokasi', 'provokatif', 'memprovokasi',
    'radikal', 'radikalisme', 'ekstremis', 'ekstremisme',
    'munafik', 'munafikin', 'hipokrit',
    'penipu', 'penipuan', 'tipu daya',
}

# ── Gabung InSet + Ekstensi Dakwah ───────────────────────────
KATA_POSITIF_FINAL = INSET_POSITIF | DAKWAH_POSITIF
KATA_NEGATIF_FINAL = INSET_NEGATIF | DAKWAH_NEGATIF

# Hapus overlap — jika kata ada di kedua kamus, hapus dari keduanya
# (kata ambigu lebih baik tidak dipakai daripada salah label)
overlap = KATA_POSITIF_FINAL & KATA_NEGATIF_FINAL
if overlap:
    KATA_POSITIF_FINAL -= overlap
    KATA_NEGATIF_FINAL -= overlap

print('── Ekstensi Domain Dakwah ──────────────────────────────')
print(f'  Tambahan Positif    : {len(DAKWAH_POSITIF)} kata domain dakwah')
print(f'  Tambahan Negatif    : {len(DAKWAH_NEGATIF)} kata domain dakwah')
print(f'  Kata overlap dihapus: {len(overlap)} kata')
print()
print('── Kamus Final ─────────────────────────────────────────')
print(f'  KATA_POSITIF_FINAL  : {len(KATA_POSITIF_FINAL):,} kata')
print(f'  KATA_NEGATIF_FINAL  : {len(KATA_NEGATIF_FINAL):,} kata')
print(f'  TOTAL               : {len(KATA_POSITIF_FINAL) + len(KATA_NEGATIF_FINAL):,} kata')
print()


# ── Fungsi Labeling ───────────────────────────────────────────
# Kata negasi yang membalik polaritas kata setelahnya
KATA_NEGASI = {
    'tidak', 'tak', 'bukan', 'jangan', 'belum', 'tanpa',
    'ga', 'gak', 'enggak', 'nggak', 'ndak', 'kagak',
}

def label_sentimen(text: str) -> str:
    """
    Labeling sentimen berbasis InSet Lexicon + ekstensi dakwah.

    Fitur:
    - Lookup O(1) menggunakan set
    - Deteksi negasi sederhana (tidak bagus → Negatif)
    - Bobot: setiap kata dihitung +1/-1, label = yang lebih banyak

    Args:
        text : teks komentar yang sudah dibersihkan (comment_clean)

    Returns:
        'Positif', 'Negatif', atau 'Netral'
    """
    if not isinstance(text, str) or not text.strip():
        return 'Netral'

    tokens    = text.lower().split()
    skor_pos  = 0
    skor_neg  = 0

    for i, token in enumerate(tokens):
        # Cek apakah token sebelumnya adalah kata negasi
        ada_negasi = (i > 0) and (tokens[i - 1] in KATA_NEGASI)

        if token in KATA_POSITIF_FINAL:
            if ada_negasi:
                skor_neg += 1   # "tidak bagus" → negatif
            else:
                skor_pos += 1
        elif token in KATA_NEGATIF_FINAL:
            if ada_negasi:
                skor_pos += 1   # "tidak buruk" → positif
            else:
                skor_neg += 1

    if skor_pos > skor_neg:
        return 'Positif'
    elif skor_neg > skor_pos:
        return 'Negatif'
    else:
        return 'Netral'


# ── Jalankan Labeling ─────────────────────────────────────────
print('=' * 60)
print('🏷️  Menjalankan labeling dengan InSet Lexicon...')
print('=' * 60)

df['sentiment_label'] = df['comment_clean'].apply(label_sentimen)

print('\n✅ Labeling selesai!')
print()
print('📊 Distribusi Label:')
dist = df['sentiment_label'].value_counts()
for label, jumlah in dist.items():
    pct = jumlah / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {label:<10} : {jumlah:>6,} ({pct:>5.1f}%) {bar}')

print()
print('📋 Contoh hasil labeling:')
display(df[['comment_text', 'comment_clean', 'sentiment_label']].sample(10, random_state=42))

📚 MEMUAT InSet Lexicon (Fachrina et al., 2016)
  ⚡ Positive Lexicon: Load dari cache Drive (inset_lexicon_positive.tsv)
  ⚡ Negative Lexicon: Load dari cache Drive (inset_lexicon_negative.tsv)

  Kata Positif (InSet) : 3,608 kata
  Kata Negatif (InSet) : 6,607 kata
  Total                : 10,215 kata

── Ekstensi Domain Dakwah ──────────────────────────────
  Tambahan Positif    : 55 kata domain dakwah
  Tambahan Negatif    : 36 kata domain dakwah
  Kata overlap dihapus: 1146 kata

── Kamus Final ─────────────────────────────────────────
  KATA_POSITIF_FINAL  : 2,502 kata
  KATA_NEGATIF_FINAL  : 5,485 kata
  TOTAL               : 7,987 kata

🏷️  Menjalankan labeling dengan InSet Lexicon...

✅ Labeling selesai!

📊 Distribusi Label:
  Positif    : 10,030 ( 53.1%) ██████████████████████████
  Negatif    :  5,177 ( 27.4%) █████████████
  Netral     :  3,698 ( 19.6%) █████████

📋 Contoh hasil labeling:


,comment_text,comment_clean,sentiment_label
2388,Mba sri ; \nMentri keuangan terbaik ❌\nMentri ...,mba sri mentri keuangan terbaik mentri majakin...,Positif
14723,Masya Allah... Terima Kasih sharingnya Bu Isah...,masya allah terima kasih sharingnya bu isah,Positif
4489,"Sehat² selalu Ustadz, keluarga dan team..\n🤲",sehat selalu ustadz keluarga dan team,Positif
14209,Assalamualaikum pak ustadz.\nLem tikus dan lem...,assalamualaikum pak ustadz lem tikus dan lem fox,Positif
4156,Assalamualaikum ustazah masyallah kangen \nSaf...,assalamualaikum ustazah masyallah kangen safak...,Positif
467,"Ustad, semoga Allah selalu jaga dan lindungi U...",ustad semoga allah selalu jaga dan lindungi us...,Positif
17678,Siapapun yg mau nyalon Gubernur di Jawabarat b...,siapapun yg mau nyalon gubernur di jawabarat b...,Negatif
7562,Assalamu'alaikum gus alhamdulillah palangkaray...,assalamu alaikum gus alhamdulillah palangkaray...,Positif
12797,"Qobiltu ijazah ya habib, semoga Habib Rifky di...",qobiltu ijazah ya habib semoga habib rifky dib...,Positif
7664,pembawa acaranya wedoke rak kudungan.. padahal...,pembawa acaranya wedoke rak kudungan padahal y...,Positif


---
## Cell 4 — Cek & Validasi Distribusi Label


In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 130})

dist = df['sentiment_label'].value_counts()
total = len(df)

print('='*55)
print('📊 DISTRIBUSI LABEL SENTIMEN')
print('='*55)
for label, n in dist.items():
    pct = n / total * 100
    print(f'  {label:<12}: {n:>6,} ({pct:.1f}%)')
print(f'  {"─"*35}')
print(f'  Total       : {total:>6,} (100.0%)')

# Cek imbalance
dominan_pct = dist.max() / total * 100
print()
if dominan_pct > 80:
    print(f'  PERINGATAN: Kelas "{dist.idxmax()}" terlalu dominan ({dominan_pct:.1f}%)')
    print('   Pertimbangkan untuk memperluas kamus kata sentimen')
    print('   atau terapkan teknik SMOTE untuk menyeimbangkan kelas')
elif dominan_pct > 60:
    print(f'  INFO: Kelas "{dist.idxmax()}" agak dominan ({dominan_pct:.1f}%) — masih wajar')
else:
    print(f' Distribusi kelas cukup seimbang!')

# Pie chart distribusi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribusi Label Sentimen', fontsize=14, fontweight='bold')

colors = {'Positif': '#55A868', 'Negatif': '#C44E52', 'Netral': '#4C72B0'}
clr = [colors.get(l, '#999') for l in dist.index]

axes[0].pie(dist.values, labels=dist.index, autopct='%1.1f%%',
           colors=clr, startangle=90,
           wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Proporsi (%)')

bars = axes[1].bar(dist.index, dist.values, color=clr, edgecolor='white', width=0.5)
axes[1].bar_label(bars, fmt='{:,.0f}', padding=5, fontsize=11, fontweight='bold')
axes[1].set_title('Jumlah Komentar per Label')
axes[1].set_ylabel('Jumlah')
axes[1].set_ylim(0, dist.max() * 1.2)

plt.tight_layout()
path_dist = os.path.join(OUTPUT_DIR, 'grafik_distribusi_label.png')
plt.savefig(path_dist, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n Grafik disimpan: {path_dist}')

# Validasi manual sampel acak
print()
print(' SAMPEL ACAK UNTUK VALIDASI MANUAL (30 komentar):')
print('   Cek apakah label yang diberikan sudah sesuai konteks')
display(df[['comment_text', 'sentiment_label']].sample(30, random_state=99))

📊 DISTRIBUSI LABEL SENTIMEN
  Positif     : 10,030 (53.1%)
  Negatif     :  5,177 (27.4%)
  Netral      :  3,698 (19.6%)
  ───────────────────────────────────
  Total       : 18,905 (100.0%)

 Distribusi kelas cukup seimbang!

 Grafik disimpan: /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/grafik_distribusi_label.png

 SAMPEL ACAK UNTUK VALIDASI MANUAL (30 komentar):
   Cek apakah label yang diberikan sudah sesuai konteks


,comment_text,sentiment_label
4600,Baru mengerti alasan hamas menyerang tgl 7 okt...,Positif
1175,Masya Allah pengen bgt bisa di ruqyah sama ustad,Positif
14065,"Ustaz llatif, SUQRON.,atas ilmu nya, sy simak ...",Positif
9166,Knpa Brata pas ngomongin judol dia diem truss ...,Negatif
10281,Assalamualaikum warahmatullahi wa barokatuh ma...,Positif
8083,Ceramah begini dikit yg nonton dikit yg komen ...,Negatif
8983,gelooo. full ngakak ni. jarang2 ampe ngakak fu...,Negatif
9902,Makasih pak kiyai atas ilmu,Positif
7238,Semoga yang like video ini dapat barokah aminn🤲🤲,Netral
4394,Ya allah.. Aamiin allahumma aamiin,Positif


---
## Cell 5 —  Preprocessing Lanjutan
> Tokenizing + Stemming menggunakan Sastrawi (algoritma Nazief & Adriani).
> Ini mengubah kata berimbuhan menjadi kata dasar.
>
> Contoh: *'menyampaikan'* → *'sampai'*, *'pembelajaran'* → *'ajar'*

In [17]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import os


if os.path.exists(PATH_STEMMED):
    # ── PATH A: File sudah ada → load langsung, skip stemming ──────────────
    print('=' * 60)
    print('⚡ RESUME TERDETEKSI — Memuat hasil stemming dari Drive...')
    print('=' * 60)

    df_stemmed = pd.read_csv(PATH_STEMMED, encoding='utf-8-sig')

    # Validasi: pastikan kolom penting ada di file yang di-load
    kolom_wajib = ['comment_stemmed', 'sentiment_label']
    kolom_hilang = [k for k in kolom_wajib if k not in df_stemmed.columns]

    if kolom_hilang:
        print(f'⚠️  File resume tidak valid, kolom hilang: {kolom_hilang}')
        print('   Hapus file clean_data_stemmed.csv lalu jalankan ulang cell ini.')
    else:
        # Sinkronisasi: ambil hanya baris yang ada di df (hasil labeling Cell 3)
        # Ini penting supaya filter dari cell sebelumnya tetap berlaku
        if 'comment_id' in df_stemmed.columns and 'comment_id' in df.columns:
            df_stemmed_only = df_stemmed[['comment_id', 'comment_stemmed']].drop_duplicates('comment_id')
            df = df.merge(df_stemmed_only, on='comment_id', how='inner')
        else:
            df = df_stemmed.copy()
        print(f'✅ Berhasil dimuat: {len(df):,} baris')
        print(f'   Kolom tersedia : {list(df.columns)}')
        print()
        print('   ℹ️  Stemming di-skip karena file sudah ada.')
        print('   ℹ️  Hapus clean_data_stemmed.csv jika ingin stemming ulang.')
        print()
        print('📋 Contoh data yang dimuat:')
        display(df[['comment_clean', 'comment_stemmed', 'sentiment_label']].head(5))

else:
    # ── PATH B: File belum ada → jalankan stemming dari awal ───────────────
    print('=' * 60)
    print('🔄 Menjalankan stemming...')
    print('   (proses ini memakan waktu ±5-30 menit tergantung jumlah data)')
    print('   Sabar ya bro, Sastrawi memproses tiap kata satu per satu!')
    print('=' * 60)

    # Inisialisasi Sastrawi
    stemmer_factory  = StemmerFactory()
    stemmer          = stemmer_factory.create_stemmer()
    sw_factory       = StopWordRemoverFactory()
    stopword_remover = sw_factory.create_stop_word_remover()

    def stemming(text: str) -> str:
        """Stemming kata berimbuhan → kata dasar (Nazief & Adriani via Sastrawi)"""
        if not isinstance(text, str) or not text.strip():
            return ''
        return stemmer.stem(text)

    def full_preprocessing(text: str) -> str:
        """Stopword removal + Stemming — untuk kolom yang akan masuk TF-IDF"""
        if not isinstance(text, str) or not text.strip():
            return ''
        text = stopword_remover.remove(text)
        text = stemmer.stem(text)
        return text

    # Tampilkan progress tiap 10%
    total     = len(df)
    milestone = max(total // 10, 1)

    hasil_stemmed = []
    kolom_input   = 'comment_no_stopword' if 'comment_no_stopword' in df.columns else 'comment_clean'
    print(f'   Menggunakan kolom input : {kolom_input}')
    print(f'   Total data              : {total:,} baris')
    print()

    for i, text in enumerate(df[kolom_input], 1):
        hasil_stemmed.append(stemming(text))
        if i % milestone == 0 or i == total:
            pct = i / total * 100
            bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
            print(f'   [{bar}] {pct:>5.1f}%  ({i:,}/{total:,})', end='\r')

    print()  # newline setelah progress bar
    df['comment_stemmed'] = hasil_stemmed

    # Hapus baris yang jadi kosong setelah stemming
    n_sebelum = len(df)
    df = df[df['comment_stemmed'].str.strip() != ''].reset_index(drop=True)
    n_sesudah = len(df)

    # 🔧 SIMPAN CHECKPOINT — Atomic write ke Drive
    tmp_path = PATH_STEMMED + '.tmp'
    df.to_csv(tmp_path, index=False, encoding='utf-8-sig')
    os.replace(tmp_path, PATH_STEMMED)

    print(f'\n✅ Stemming selesai!')
    print(f'   Data sebelum : {n_sebelum:,}')
    print(f'   Data sesudah : {n_sesudah:,} ({n_sebelum - n_sesudah} baris kosong dihapus)')
    print(f'   💾 Checkpoint disimpan: {PATH_STEMMED}')
    print(f'      → Jika Colab disconnect, jalankan ulang cell ini')
    print(f'        untuk load dari file tanpa stemming ulang.')
    print()
    print('📋 Contoh hasil stemming:')
    display(df[['comment_clean', 'comment_stemmed', 'sentiment_label']].head(8))

⚡ RESUME TERDETEKSI — Memuat hasil stemming dari Drive...
✅ Berhasil dimuat: 18,896 baris
   Kolom tersedia : ['comment_id', 'comment_text', 'author_name', 'like_count', 'published_at', 'video_id', 'channel_name', 'topik_utama', 'panjang_raw', 'jumlah_kata_raw', 'comment_clean', 'comment_no_stopword', 'panjang_clean', 'jumlah_kata_clean', 'tokens', 'sentiment_label', 'comment_stemmed']

   ℹ️  Stemming di-skip karena file sudah ada.
   ℹ️  Hapus clean_data_stemmed.csv jika ingin stemming ulang.

📋 Contoh data yang dimuat:


,comment_clean,comment_stemmed,sentiment_label
0,andai saja hadir juga aa gym buya yahya habib ...,andai hadir aa gym buya yahya habib riziek ust...,Positif
1,saya terlahir nu dan tumbuh dg segala aturan d...,lahir nu tumbuh dg atur tata nu berkat ustadz ...,Positif
2,yang nonton ini sambil nangiss allahumma sholl...,nonton nangiss allahumma sholli ala sayyidina ...,Netral
3,saya bersaksi guru guru kami adalah orang oran...,saksi guru guru soleh panjang umur mulia dunia...,Positif
4,teringat caption pada foto para guru ini bbrp ...,caption foto guru bbrp tahun apa halaqah mu sa...,Positif


---
## Cell 6 — TF-IDF Feature Extraction
> Mengubah teks menjadi representasi angka (vektor) yang bisa dipahami algoritma.
>
> **Analogi:** TF-IDF seperti mengubah setiap komentar menjadi 'sidik jari angka'
> yang merepresentasikan kata-kata paling penting di dalamnya.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Kolom teks yang akan dijadikan fitur
KOLOM_FITUR = 'comment_stemmed'
KOLOM_LABEL = 'sentiment_label'

# Inisialisasi TF-IDF
tfidf = TfidfVectorizer(
    max_features = 5000,    # Ambil 5000 kata dengan bobot tertinggi
    min_df       = 2,       # Kata harus muncul minimal di 2 dokumen
    max_df       = 0.95,    # Abaikan kata yang muncul di >95% dokumen
    ngram_range  = (1, 2),  # Gunakan unigram dan bigram
    sublinear_tf = True,    # Gunakan log normalization untuk TF
)

X = tfidf.fit_transform(df[KOLOM_FITUR].fillna(''))
y = df[KOLOM_LABEL]

print('='*55)
print('📐 HASIL TF-IDF VECTORIZATION')
print('='*55)
print(f'Jumlah dokumen (komentar) : {X.shape[0]:,}')
print(f'Jumlah fitur (kata/bigram): {X.shape[1]:,}')
print(f'Ukuran matriks TF-IDF     : {X.shape}')
print()
print('Top 20 kata dengan bobot TF-IDF tertinggi:')
fitur_names = tfidf.get_feature_names_out()
print(', '.join(fitur_names[:20]))

📐 HASIL TF-IDF VECTORIZATION
Jumlah dokumen (komentar) : 18,896
Jumlah fitur (kata/bigram): 5,000
Ukuran matriks TF-IDF     : (18896, 5000)

Top 20 kata dengan bobot TF-IDF tertinggi:
aa, aa gim, aa gym, aa hilman, aa moga, aa sehat, aaamiiin, aaamiin, aaamiinn, aagym, aalamiin, aalamin, aali, aali sayyidina, aalloh, aam, aamiiiin, aamiiin, aamiin, aamiin aamiin


---
## Cell 7 — Split Data Train / Test
> Membagi dataset menjadi **80% data latih** dan **20% data uji**.
>
> **Analogi:** Seperti latihan soal (80%) dan ujian sebenarnya (20%).
> Model belajar dari data latih, lalu diuji pada data yang belum pernah dilihat.

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,         # 20% untuk testing
    random_state = 42,          # Seed agar hasil reproducible
    stratify     = y            # Pastikan proporsi kelas sama di train & test
)

print('='*55)
print('  HASIL SPLIT DATA')
print('='*55)
print(f'Total data          : {X.shape[0]:,}')
print(f'Data latih (80%)    : {X_train.shape[0]:,}')
print(f'Data uji   (20%)    : {X_test.shape[0]:,}')
print()
print('Distribusi kelas di data latih:')
for label, n in y_train.value_counts().items():
    print(f'  {label:<12}: {n:,} ({n/len(y_train)*100:.1f}%)')
print()
print('Distribusi kelas di data uji:')
for label, n in y_test.value_counts().items():
    print(f'  {label:<12}: {n:,} ({n/len(y_test)*100:.1f}%)')

  HASIL SPLIT DATA
Total data          : 18,896
Data latih (80%)    : 15,116
Data uji   (20%)    : 3,780

Distribusi kelas di data latih:
  Positif     : 8,023 (53.1%)
  Negatif     : 4,140 (27.4%)
  Netral      : 2,953 (19.5%)

Distribusi kelas di data uji:
  Positif     : 2,007 (53.1%)
  Negatif     : 1,035 (27.4%)
  Netral      : 738 (19.5%)


---
## Cell 8 — Training Multinomial Naive Bayes
> Training model klasifikasi sentimen menggunakan Multinomial Naive Bayes.
>
> Parameter **alpha** adalah Laplace Smoothing — mencegah probabilitas bernilai nol
> untuk kata yang belum pernah muncul di data latih.

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import time

print(' Training Multinomial Naive Bayes...')
start_time = time.time()

# Inisialisasi model dengan Laplace Smoothing (alpha=1.0)
nb_model = MultinomialNB(alpha=1.0)  # alpha=1.0 = Laplace Smoothing

# Training
nb_model.fit(X_train, y_train)

elapsed = time.time() - start_time
print(f'\n Training selesai dalam {elapsed:.2f} detik!')
print(f'   Algoritma  : Multinomial Naive Bayes')
print(f'   Alpha      : {nb_model.alpha} (Laplace Smoothing)')
print(f'   Kelas      : {list(nb_model.classes_)}')
print(f'   Fitur      : {nb_model.n_features_in_:,}')
print()

# Prediksi pada data uji
y_pred = nb_model.predict(X_test)
y_prob = nb_model.predict_proba(X_test)

print(' Contoh 10 hasil prediksi:')
# Ambil indeks asli dari df
idx_test = y_test.index[:10]
contoh = pd.DataFrame({
    'Komentar'     : df.loc[idx_test, 'comment_text'].values,
    'Label Asli'   : y_test.values[:10],
    'Label Prediksi': y_pred[:10],
    'Benar?'       : ['✅' if a==p else '❌'
                      for a,p in zip(y_test.values[:10], y_pred[:10])]
})
display(contoh)

 Training Multinomial Naive Bayes...

 Training selesai dalam 0.08 detik!
   Algoritma  : Multinomial Naive Bayes
   Alpha      : 1.0 (Laplace Smoothing)
   Kelas      : [np.str_('Negatif'), np.str_('Netral'), np.str_('Positif')]
   Fitur      : 5,000

 Contoh 10 hasil prediksi:


,Komentar,Label Asli,Label Prediksi,Benar?
0,Ya Allah ya rohman ya rohim jadikanlah ustaz b...,Positif,Positif,✅
1,alhamdulilah...sy mendapatkan ilmu dr ustad us...,Netral,Positif,❌
2,Suaranya pas doa gak jelas ya Alloh,Positif,Positif,✅
3,bgmn cara menghilangkan sampah pikiran dan per...,Negatif,Negatif,✅
4,bismillahirrahmanirrahim semoga aku rajin shal...,Positif,Positif,✅
5,anak saya.saya namakan ahmad FAIZAR bawazier s...,Negatif,Negatif,✅
6,"Mikrofon nya buruk tuh, panitia perbaiki posi...",Netral,Negatif,❌
7,Assalamu'alaikum wr wb Ustadz,Positif,Positif,✅
8,Dari plat AG .....PBNU ini klo dibiarkan bisa2...,Netral,Negatif,❌
9,doa istikharahnya apa ya?,Netral,Positif,❌


---
## Cell 9 —  Evaluasi Model: Confusion Matrix + Classification Report
> Mengukur performa model menggunakan 4 metrik utama:
> **Accuracy, Precision, Recall, F1-Score**


In [21]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ── Hitung semua metrik ───────────────────────────────────────
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
cm   = confusion_matrix(y_test, y_pred, labels=['Positif', 'Negatif', 'Netral'])

print('='*55)
print(' HASIL EVALUASI MODEL NAIVE BAYES')
print('='*55)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}  ({prec*100:.2f}%)')
print(f'  Recall    : {rec:.4f}  ({rec*100:.2f}%)')
print(f'  F1-Score  : {f1:.4f}  ({f1*100:.2f}%)')
print()

# Penilaian performa
if acc >= 0.85:
    print(f'🏆 Performa SANGAT BAIK! (Accuracy ≥ 85%)')
elif acc >= 0.75:
    print(f'✅ Performa BAIK (Accuracy ≥ 75%)')
elif acc >= 0.65:
    print(f'  Performa CUKUP (Accuracy ≥ 65%) — pertimbangkan optimasi')
else:
    print(f'❌ Performa KURANG (Accuracy < 65%) — perlu optimasi')

print()
print('─'*55)
print('CLASSIFICATION REPORT (per kelas):')
print('─'*55)
print(classification_report(
    y_test, y_pred,
    labels=['Positif', 'Negatif', 'Netral'],
    target_names=['Positif', 'Negatif', 'Netral'],
    zero_division=0
))

print('─'*55)
print('CONFUSION MATRIX:')
print('─'*55)
print('(Baris = Label Asli, Kolom = Label Prediksi)')
print(f'{"":12} Positif  Negatif  Netral')
for i, label in enumerate(['Positif', 'Negatif', 'Netral']):
    print(f'{label:<12} {cm[i][0]:>7}  {cm[i][1]:>7}  {cm[i][2]:>6}')

 HASIL EVALUASI MODEL NAIVE BAYES
  Accuracy  : 0.6672  (66.72%)
  Precision : 0.6360  (63.60%)
  Recall    : 0.6672  (66.72%)
  F1-Score  : 0.6253  (62.53%)

  Performa CUKUP (Accuracy ≥ 65%) — pertimbangkan optimasi

───────────────────────────────────────────────────────
CLASSIFICATION REPORT (per kelas):
───────────────────────────────────────────────────────
              precision    recall  f1-score   support

     Positif       0.71      0.86      0.78      2007
     Negatif       0.61      0.69      0.65      1035
      Netral       0.47      0.11      0.18       738

    accuracy                           0.67      3780
   macro avg       0.60      0.55      0.54      3780
weighted avg       0.64      0.67      0.63      3780

───────────────────────────────────────────────────────
CONFUSION MATRIX:
───────────────────────────────────────────────────────
(Baris = Label Asli, Kolom = Label Prediksi)
             Positif  Negatif  Netral
Positif         1720      230      57
Ne

---
## Cell 10 — Visualisasi Hasil Evaluasi
> Grafik confusion matrix dan perbandingan metrik — siap untuk dijadikan gambar di Bab 4!

In [22]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Evaluasi Model Multinomial Naive Bayes\n'
    'Analisis Sentimen Dakwah Digital Kota Bandung',
    fontsize=13, fontweight='bold', y=1.02
)

LABEL_KELAS = ['Positif', 'Negatif', 'Netral']

# ── GRAFIK 1: Confusion Matrix ───────────────────────────────
ax1 = axes[0]
sns.heatmap(
    cm,
    annot      = True,
    fmt        = 'd',
    cmap       = 'Blues',
    xticklabels= LABEL_KELAS,
    yticklabels= LABEL_KELAS,
    linewidths = 0.5,
    ax         = ax1,
    cbar_kws   = {'label': 'Jumlah'}
)
ax1.set_title('Confusion Matrix', fontweight='bold', pad=10)
ax1.set_xlabel('Label Prediksi')
ax1.set_ylabel('Label Asli')

# ── GRAFIK 2: Metrik per kelas ───────────────────────────────
from sklearn.metrics import precision_recall_fscore_support
prec_per, rec_per, f1_per, sup_per = precision_recall_fscore_support(
    y_test, y_pred, labels=LABEL_KELAS, zero_division=0
)

ax2 = axes[1]
x    = np.arange(len(LABEL_KELAS))
w    = 0.25
b1   = ax2.bar(x - w,   prec_per, w, label='Precision', color='#4C72B0', edgecolor='white')
b2   = ax2.bar(x,       rec_per,  w, label='Recall',    color='#55A868', edgecolor='white')
b3   = ax2.bar(x + w,   f1_per,   w, label='F1-Score',  color='#C44E52', edgecolor='white')
ax2.bar_label(b1, fmt='%.2f', padding=2, fontsize=8)
ax2.bar_label(b2, fmt='%.2f', padding=2, fontsize=8)
ax2.bar_label(b3, fmt='%.2f', padding=2, fontsize=8)
ax2.set_xticks(x)
ax2.set_xticklabels(LABEL_KELAS)
ax2.set_ylim(0, 1.2)
ax2.set_title('Precision / Recall / F1 per Kelas', fontweight='bold', pad=10)
ax2.set_ylabel('Skor')
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.4)

# ── GRAFIK 3: Ringkasan metrik keseluruhan ───────────────────
ax3 = axes[2]
metrik_nama  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrik_nilai = [acc, prec, rec, f1]
warna        = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = ax3.bar(metrik_nama, metrik_nilai, color=warna, edgecolor='white', width=0.5)
ax3.bar_label(bars, fmt='%.4f', padding=5, fontsize=11, fontweight='bold')
ax3.set_ylim(0, 1.2)
ax3.set_title('Metrik Evaluasi Keseluruhan', fontweight='bold', pad=10)
ax3.set_ylabel('Skor')
ax3.axhline(y=0.85, color='red', linestyle='--', alpha=0.6, label='Target 85%')
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.4)

plt.tight_layout()
path_eval = os.path.join(OUTPUT_DIR, 'grafik_evaluasi_naive_bayes.png')
plt.savefig(path_eval, bbox_inches='tight', facecolor='white', dpi=150)
plt.show()
print(f'\n💾 Grafik disimpan: {path_eval}')

# ── GRAFIK 4: Confusion Matrix Normalized (%) ────────────────
fig2, ax4 = plt.subplots(figsize=(7, 6))
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(
    cm_norm,
    annot      = True,
    fmt        = '.2%',
    cmap       = 'Greens',
    xticklabels= LABEL_KELAS,
    yticklabels= LABEL_KELAS,
    linewidths = 0.5,
    ax         = ax4,
)
ax4.set_title(
    'Confusion Matrix (Normalized / %)',
    fontweight='bold', pad=10
)
ax4.set_xlabel('Label Prediksi')
ax4.set_ylabel('Label Asli')
plt.tight_layout()
path_cm_norm = os.path.join(OUTPUT_DIR, 'grafik_confusion_matrix_normalized.png')
plt.savefig(path_cm_norm, bbox_inches='tight', facecolor='white', dpi=150)
plt.show()
print(f'💾 Grafik disimpan: {path_cm_norm}')


💾 Grafik disimpan: /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/grafik_evaluasi_naive_bayes.png
💾 Grafik disimpan: /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/grafik_confusion_matrix_normalized.png


---
## Cell 11 — Simpan Model & Hasil
> Simpan model yang sudah dilatih, hasil prediksi lengkap, dan laporan akhir
> ke Google Drive supaya bisa dipakai lagi tanpa training ulang.

In [23]:
import pickle
import json

# ── 1. Simpan model Naive Bayes ──────────────────────────────
path_model = os.path.join(OUTPUT_DIR, 'model_naive_bayes.pkl')
with open(path_model, 'wb') as f:
    pickle.dump(nb_model, f)
print(f'✅ Model disimpan   : {path_model}')

# ── 2. Simpan TF-IDF vectorizer ──────────────────────────────
path_tfidf = os.path.join(OUTPUT_DIR, 'tfidf_vectorizer.pkl')
with open(path_tfidf, 'wb') as f:
    pickle.dump(tfidf, f)
print(f'✅ TF-IDF disimpan  : {path_tfidf}')

# ── 3. Simpan hasil prediksi lengkap ke CSV ──────────────────
df_hasil                  = df.copy()
df_hasil['split']         = 'train'
df_hasil.loc[y_test.index, 'split'] = 'test'
df_hasil['label_prediksi'] = np.nan
df_hasil.loc[y_test.index, 'label_prediksi'] = y_pred

path_hasil = os.path.join(OUTPUT_DIR, 'hasil_prediksi_lengkap.csv')
df_hasil.to_csv(path_hasil, index=False, encoding='utf-8-sig')
print(f'✅ Hasil prediksi   : {path_hasil}')

# ── 4. Simpan ringkasan metrik ke JSON ───────────────────────
ringkasan = {
    'model'           : 'Multinomial Naive Bayes',
    'alpha'           : float(nb_model.alpha),
    'total_data'      : int(len(df)),
    'data_latih'      : int(X_train.shape[0]),
    'data_uji'        : int(X_test.shape[0]),
    'jumlah_fitur'    : int(X.shape[1]),
    'accuracy'        : round(float(acc),  4),
    'precision'       : round(float(prec), 4),
    'recall'          : round(float(rec),  4),
    'f1_score'        : round(float(f1),   4),
    'distribusi_label': df['sentiment_label'].value_counts().to_dict(),
}
path_json = os.path.join(OUTPUT_DIR, 'ringkasan_hasil.json')
with open(path_json, 'w', encoding='utf-8') as f:
    json.dump(ringkasan, f, ensure_ascii=False, indent=2)
print(f'✅ Ringkasan metrik : {path_json}')

# ── 5. Laporan akhir ─────────────────────────────────────────
print()
print('='*60)
print(' SCRIPT 3 SELESAI!')
print('='*60)
print(f'  Model     : Multinomial Naive Bayes (alpha=1.0)')
print(f'  Data uji  : {X_test.shape[0]:,} komentar')
print()
print(f'  ACCURACY  : {acc*100:.2f}%')
print(f'  PRECISION : {prec*100:.2f}%')
print(f'  RECALL    : {rec*100:.2f}%')
print(f'  F1-SCORE  : {f1*100:.2f}%')
print()
print('  📁 Semua file tersimpan di:')
print(f'  {OUTPUT_DIR}/')
print()
print('  ✍️  Angka-angka di atas siap ditulis di Bab 4 skripsi!')
print('='*60)

✅ Model disimpan   : /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/model_naive_bayes.pkl
✅ TF-IDF disimpan  : /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/tfidf_vectorizer.pkl
✅ Hasil prediksi   : /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/hasil_prediksi_lengkap.csv
✅ Ringkasan metrik : /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/ringkasan_hasil.json

 SCRIPT 3 SELESAI!
  Model     : Multinomial Naive Bayes (alpha=1.0)
  Data uji  : 3,780 komentar

  ACCURACY  : 66.72%
  PRECISION : 63.60%
  RECALL    : 66.72%
  F1-SCORE  : 62.53%

  📁 Semua file tersimpan di:
  /content/drive/MyDrive/skripsi_dakwah_bandung/hasil_model/

  ✍️  Angka-angka di atas siap ditulis di Bab 4 skripsi!


---
## Cell 12 — Dataset Final yang Rapi & Lengkap
> Cell ini menghasilkan **2 dataset final** yang hanya berisi kolom yang
> benar-benar diperlukan — siap dijadikan lampiran skripsi.
>
> **Output:**
> - `dataset_final_komentar.csv` → semua komentar + label + hasil prediksi
> - `dataset_final_video.csv` → semua video + topik + ringkasan + distribusi sentimen
> - `dataset_final_per_penceramah.csv` → rekap sentimen per penceramah (untuk Bab 4)

In [24]:
import pandas as pd
import numpy as np

# ============================================================
# DATASET FINAL 1: KOMENTAR
# Kolom yang benar-benar diperlukan saja
# ============================================================

# Susun kolom yang diperlukan dari df (yang sudah ada label & prediksi)
# Pastikan kolom label_prediksi dan prediksi_benar sudah ada dari Cell 8
df_final_komentar = df.copy()

# Tambahkan kolom prediksi dari hasil Cell 8
df_final_komentar['label_prediksi'] = np.nan
df_final_komentar['prediksi_benar'] = np.nan
df_final_komentar.loc[y_test.index, 'label_prediksi'] = y_pred
df_final_komentar.loc[y_test.index, 'prediksi_benar'] = (
    y_test.values == y_pred
)
df_final_komentar['split'] = 'train'
df_final_komentar.loc[y_test.index, 'split'] = 'test'

# Pilih HANYA kolom yang diperlukan
KOLOM_KOMENTAR = [
    'channel_name',        # Nama penceramah
    'topik_utama',         # Topik dakwah video
    'comment_text',        # Komentar asli (untuk verifikasi)
    'comment_clean',       # Teks bersih (input TF-IDF)
    'comment_no_stopword', # Teks tanpa stopword (input NB)
    'sentiment_label',     # Label hasil lexicon
    'split',               # train / test
    'label_prediksi',      # Hasil prediksi model NB
    'prediksi_benar',      # Benar/salah prediksi
]

# Hanya ambil kolom yang ada
kolom_ada = [k for k in KOLOM_KOMENTAR if k in df_final_komentar.columns]
df_final_komentar = df_final_komentar[kolom_ada].reset_index(drop=True)

# Simpan
path_final_k = os.path.join(OUTPUT_DIR, 'dataset_final_komentar.csv')
df_final_komentar.to_csv(path_final_k, index=False, encoding='utf-8-sig')

print('='*60)
print('✅ DATASET FINAL 1: KOMENTAR')
print('='*60)
print(f'Jumlah baris   : {len(df_final_komentar):,}')
print(f'Jumlah kolom   : {len(df_final_komentar.columns)}')
print(f'Kolom          : {list(df_final_komentar.columns)}')
print()
print('Preview 5 baris pertama:')
display(df_final_komentar.head())
print()
print(f'Distribusi sentiment_label:')
for label, n in df_final_komentar['sentiment_label'].value_counts().items():
    pct = n / len(df_final_komentar) * 100
    print(f'  {label:<12}: {n:>6,} ({pct:.1f}%)')
print()
print(f'Distribusi split (train/test):')
print(df_final_komentar['split'].value_counts().to_string())

# ============================================================
# DATASET FINAL 2: VIDEO
# ============================================================

# Load clean_data_video dari Google Drive
path_video = os.path.join(
    '/content/drive/MyDrive/skripsi_dakwah_bandung',
    'clean_data_video.csv'
)

df_video_clean = pd.read_csv(path_video, encoding='utf-8-sig')

# Hitung distribusi sentimen per video
# Gabungkan dengan data komentar berdasarkan channel_name + topik_utama
sentimen_per_video = df_final_komentar.groupby(
    ['channel_name', 'topik_utama', 'sentiment_label']
).size().unstack(fill_value=0).reset_index()

# Pastikan semua kolom sentimen ada
for kelas in ['Positif', 'Negatif', 'Netral']:
    if kelas not in sentimen_per_video.columns:
        sentimen_per_video[kelas] = 0

sentimen_per_video['total_komentar'] = (
    sentimen_per_video['Positif'] +
    sentimen_per_video['Negatif'] +
    sentimen_per_video['Netral']
)
sentimen_per_video['pct_positif'] = (
    sentimen_per_video['Positif'] / sentimen_per_video['total_komentar'] * 100
).round(1)
sentimen_per_video['pct_negatif'] = (
    sentimen_per_video['Negatif'] / sentimen_per_video['total_komentar'] * 100
).round(1)
sentimen_per_video['pct_netral'] = (
    sentimen_per_video['Netral'] / sentimen_per_video['total_komentar'] * 100
).round(1)
sentimen_per_video['sentimen_dominan'] = sentimen_per_video[
    ['Positif', 'Negatif', 'Netral']
].idxmax(axis=1)

# Pilih kolom video yang diperlukan
KOLOM_VIDEO = [
    'channel_name',
    'title',
    'topik_utama',
    'ringkasan_konten',
    'kesimpulan',
    'published_at',
]
kolom_video_ada = [k for k in KOLOM_VIDEO if k in df_video_clean.columns]
df_video_final = df_video_clean[kolom_video_ada].copy()

# Gabungkan dengan distribusi sentimen
df_video_final = df_video_final.merge(
    sentimen_per_video[[
        'channel_name', 'topik_utama',
        'Positif', 'Negatif', 'Netral',
        'total_komentar',
        'pct_positif', 'pct_negatif', 'pct_netral',
        'sentimen_dominan'
    ]],
    on=['channel_name', 'topik_utama'],
    how='left'
)

path_final_v = os.path.join(OUTPUT_DIR, 'dataset_final_video.csv')
df_video_final.to_csv(path_final_v, index=False, encoding='utf-8-sig')

print()
print('='*60)
print('✅ DATASET FINAL 2: VIDEO + DISTRIBUSI SENTIMEN')
print('='*60)
print(f'Jumlah baris   : {len(df_video_final):,}')
print(f'Jumlah kolom   : {len(df_video_final.columns)}')
print(f'Kolom          : {list(df_video_final.columns)}')
print()
print('Preview:')
display(df_video_final[[
    'channel_name','title','topik_utama',
    'total_komentar','Positif','Negatif','Netral',
    'sentimen_dominan'
]])

# ============================================================
# DATASET FINAL 3: REKAP PER PENCERAMAH
# Ini yang paling berguna untuk Bab 4!
# ============================================================

rekap = df_final_komentar.groupby('channel_name').agg(
    total_komentar   = ('comment_text', 'count'),
    jumlah_positif   = ('sentiment_label', lambda x: (x == 'Positif').sum()),
    jumlah_negatif   = ('sentiment_label', lambda x: (x == 'Negatif').sum()),
    jumlah_netral    = ('sentiment_label', lambda x: (x == 'Netral').sum()),
).reset_index()

rekap['pct_positif']     = (rekap['jumlah_positif'] / rekap['total_komentar'] * 100).round(1)
rekap['pct_negatif']     = (rekap['jumlah_negatif'] / rekap['total_komentar'] * 100).round(1)
rekap['pct_netral']      = (rekap['jumlah_netral']  / rekap['total_komentar'] * 100).round(1)
rekap['sentimen_dominan'] = rekap[['jumlah_positif','jumlah_negatif','jumlah_netral']].idxmax(axis=1).str.replace('jumlah_','')
rekap['sentimen_dominan'] = rekap['sentimen_dominan'].str.capitalize()

# Urutkan berdasarkan pct_positif tertinggi
rekap = rekap.sort_values('pct_positif', ascending=False).reset_index(drop=True)
rekap.index += 1  # Mulai dari 1

path_rekap = os.path.join(OUTPUT_DIR, 'dataset_final_per_penceramah.csv')
rekap.to_csv(path_rekap, encoding='utf-8-sig')

print()
print('='*60)
print('✅ DATASET FINAL 3: REKAP SENTIMEN PER PENCERAMAH')
print('='*60)
print(f'Jumlah penceramah: {len(rekap)}')
print()
print('Tabel rekap (urut % positif tertinggi):')
display(rekap)

# ============================================================
# RINGKASAN SEMUA OUTPUT
# ============================================================
print()
print('='*60)
print('📦 SEMUA DATASET FINAL TERSIMPAN')
print('='*60)
print(f'1. dataset_final_komentar.csv    → {len(df_final_komentar):,} baris, {len(df_final_komentar.columns)} kolom')
print(f'2. dataset_final_video.csv       → {len(df_video_final):,} baris, {len(df_video_final.columns)} kolom')
print(f'3. dataset_final_per_penceramah.csv → {len(rekap):,} baris (1 per penceramah)')
print()
print('💡 Ketiga file ini siap dijadikan lampiran skripsi!')
print('   dataset_final_per_penceramah.csv → langsung masuk Bab 4')


✅ DATASET FINAL 1: KOMENTAR
Jumlah baris   : 18,896
Jumlah kolom   : 9
Kolom          : ['channel_name', 'topik_utama', 'comment_text', 'comment_clean', 'comment_no_stopword', 'sentiment_label', 'split', 'label_prediksi', 'prediksi_benar']

Preview 5 baris pertama:


,channel_name,topik_utama,comment_text,comment_clean,comment_no_stopword,sentiment_label,split,label_prediksi,prediksi_benar
0,Ustadz Adi Hidayat,Akhlak,"Andai saja hadir juga : Aa Gym, Buya Yahya, Ha...",andai saja hadir juga aa gym buya yahya habib ...,andai hadir aa gym buya yahya habib riziek ust...,Positif,train,NaN,NaN
1,Ustadz Adi Hidayat,Akhlak,Saya terlahir NU dan tumbuh dg segala aturan d...,saya terlahir nu dan tumbuh dg segala aturan d...,terlahir nu tumbuh dg aturan tata nu berkat us...,Positif,train,NaN,NaN
2,Ustadz Adi Hidayat,Akhlak,YANG NONTON INI SAMBIL NANGISS😭☝🏻☝🏻☝🏻 Allahumm...,yang nonton ini sambil nangiss allahumma sholl...,nonton nangiss allahumma sholli ala sayyidina ...,Netral,train,NaN,NaN
3,Ustadz Adi Hidayat,Akhlak,saya bersaksi guru guru kami adalah orang oran...,saya bersaksi guru guru kami adalah orang oran...,bersaksi guru guru soleh panjangkan umur mulia...,Positif,test,Positif,True
4,Ustadz Adi Hidayat,Akhlak,Teringat caption pada foto para guru ini bbrp ...,teringat caption pada foto para guru ini bbrp ...,caption foto guru bbrp tahun apapun halaqah mu...,Positif,train,NaN,NaN



Distribusi sentiment_label:
  Positif     : 10,030 (53.1%)
  Negatif     :  5,175 (27.4%)
  Netral      :  3,691 (19.5%)

Distribusi split (train/test):
split
train    15116
test      3780

✅ DATASET FINAL 2: VIDEO + DISTRIBUSI SENTIMEN
Jumlah baris   : 162
Jumlah kolom   : 14
Kolom          : ['channel_name', 'title', 'topik_utama', 'ringkasan_konten', 'kesimpulan', 'published_at', 'Positif', 'Negatif', 'Netral', 'total_komentar', 'pct_positif', 'pct_negatif', 'pct_netral', 'sentimen_dominan']

Preview:


,channel_name,title,topik_utama,total_komentar,Positif,Negatif,Netral,sentimen_dominan
0,Ustadz Adi Hidayat,Kunjungan para Ustadz Sahabat UAH - Ustadz Adi...,Pendidikan Islam,NaN,NaN,NaN,NaN,NaN
1,Ustadz Adi Hidayat,UAH Dilantik Jadi Stafsus? Hoax Atau Nyata? - ...,"Klarifikasi, Peran Ustadz, Prinsip Dakwah",NaN,NaN,NaN,NaN,NaN
2,Ustadz Adi Hidayat,Perbaiki Shalatmu Maka Allah Akan Permudah Uru...,Fiqih,NaN,NaN,NaN,NaN,NaN
3,Ustadz Muhammad Faizar Hidayatullah,PENAMPAKAN POCONG DI RUMAH PANJI PETUALANG,Akidah,890.0,345.0,417.0,128.0,Negatif
4,Ustadz Muhammad Faizar Hidayatullah,MANTAN AHLI SANTET - RIA PUSPITA | PERNAH MAKA...,Pengalaman Mantan Ahli Santet,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
157,Ustadz Zacky Mirza,Pengajian Bulanan MEWUJUDKAN GENERASI RABBANI ...,"Generasi Rabbani, Ramadan, Suri Tauladan",20.0,8.0,3.0,9.0,Netral
158,Ustadz Zacky Mirza,SABTU 4 APRIL 2026 - pukul 12.30 WIB_ KH.Syari...,Akhlak,2.0,1.0,0.0,1.0,Positif
159,Ustadz Subhan Bawazier,Cuma Titipan | Ustadz Subhan Bawazier,Dunia Hanyalah Titipan Allah,NaN,NaN,NaN,NaN,NaN
160,Dr. Zaidul Akbar (Jurus Sehat Rasulullah),Makanan Yang Tidak Menjadi Beban - dr.Zadul Ak...,Kesehatan Islam,27.0,16.0,7.0,4.0,Positif



✅ DATASET FINAL 3: REKAP SENTIMEN PER PENCERAMAH
Jumlah penceramah: 54

Tabel rekap (urut % positif tertinggi):


,channel_name,total_komentar,jumlah_positif,jumlah_negatif,jumlah_netral,pct_positif,pct_negatif,pct_netral,sentimen_dominan
1,Ustadz Erick Yusuf,3,3,0,0,100.0,0.0,0.0,Positif
2,Ustadz Aam Amiruddin (Percikan Iman),43,36,6,1,83.7,14.0,2.3,Positif
3,Ustadz Handy Bonny,418,317,32,69,75.8,7.7,16.5,Positif
4,Ustadz Abdul Somad (UAS),772,584,71,117,75.6,9.2,15.2,Positif
5,Ustadzah Irena Handono,669,484,102,83,72.3,15.2,12.4,Positif
6,Bunda Imas Karyamah,40,28,4,8,70.0,10.0,20.0,Positif
7,Ustadz Subhan Bawazier,40,28,8,4,70.0,20.0,10.0,Positif
8,Ustadzah Qotrunnada,573,399,92,82,69.6,16.1,14.3,Positif
9,Buya Yahya (Al-Bahjah),531,361,77,93,68.0,14.5,17.5,Positif
10,Ustadz Ucu Najmudin,82,55,11,16,67.1,13.4,19.5,Positif



📦 SEMUA DATASET FINAL TERSIMPAN
1. dataset_final_komentar.csv    → 18,896 baris, 9 kolom
2. dataset_final_video.csv       → 162 baris, 14 kolom
3. dataset_final_per_penceramah.csv → 54 baris (1 per penceramah)

💡 Ketiga file ini siap dijadikan lampiran skripsi!
   dataset_final_per_penceramah.csv → langsung masuk Bab 4
